<a href="https://colab.research.google.com/github/mahidodiya/evolve/blob/main/skill_check_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
print("GPU available:", torch.cuda.is_available())
!nvidia-smi

GPU available: True
Tue Oct 13 12:13:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             16W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------------------

In [2]:
!pip install -q transformers accelerate bitsandbytes
!pip install -q "qwen-omni-utils[decord]" pymupdf python-docx pandas
# Optional (only if you set SHOW_EMBED = True below):
# !pip install -q sentence-transformers scikit-learn
# If the Qwen2.5-Omni import fails, use the dev build instead:
# !pip install -q git+https://github.com/huggingface/transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 79.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 107.6 MB/s eta 0:00:00


In [1]:
import os, gc, glob, re, time, hashlib, subprocess, math
from concurrent.futures import ThreadPoolExecutor
import torch, pymupdf, docx
import numpy as np, pandas as pd
from transformers import Qwen2_5OmniForConditionalGeneration, Qwen2_5OmniProcessor, BitsAndBytesConfig
from qwen_omni_utils import process_mm_info

assert torch.cuda.is_available(), "No GPU: set the runtime type to T4 GPU and restart."

model_id = "Qwen/Qwen2.5-Omni-3B"
processor = Qwen2_5OmniProcessor.from_pretrained(model_id)
processor.tokenizer.padding_side = "left"      # required for batched generation and last-token logits

def load(quantize_4bit=False):
    kwargs = dict(torch_dtype=torch.float16, device_map={"": 0}, attn_implementation="sdpa")
    if quantize_4bit:
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4",
            llm_int8_skip_modules=["audio_tower", "visual", "lm_head"])
    return Qwen2_5OmniForConditionalGeneration.from_pretrained(model_id, **kwargs)

try:
    model = load(quantize_4bit=False); print("Loaded fp16")
except torch.cuda.OutOfMemoryError:
    gc.collect(); torch.cuda.empty_cache()
    model = load(quantize_4bit=True); print("fp16 didn't fit, loaded 4-bit")

if hasattr(model, "disable_talker"):      # text output only: free the speech modules
    model.disable_talker()
thinker = model.thinker
torch.cuda.empty_cache()
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

[transformers] Model config: tts_text_start_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151860. This may result in unexpected behavior.
[transformers] Model config: tts_text_end_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151861. This may result in unexpected behavior.
[transformers] Model config: tts_text_pad_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151859. This may result in unexpected behavior.
[transformers] Model config: vision_start_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151652. This may result in unexpected behavior.
[transformers] Model config: vision_end_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151653. This may result in unexpected behavior.
[transformers] Model config: audio_start_token_id must be `None` or an integer within the vocabulary (between 0 and 8447

Loading weights:   0%|          | 0/2543 [00:00<?, ?it/s]

[transformers] Qwen2_5OmniForConditionalGeneration LOAD REPORT from: Qwen/Qwen2.5-Omni-3B
Key                                                | Status     |  | 
---------------------------------------------------+------------+--+-
token2wav.code2wav_dit_model.rotary_embed.inv_freq | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded fp16
GPU memory used: 9.4 GB


In [22]:
IMG = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".gif"}
VID = {".mp4", ".mov", ".avi", ".mkv", ".webm"}
AUD = {".wav", ".mp3", ".m4a", ".flac", ".ogg"}
TXT = {".txt", ".md", ".csv", ".json", ".html"}

# ---- text / PDF ----
MAX_CHARS, MAX_PIECES = 12000, 8       # long text: summarize up to MAX_PIECES pieces spread across the document

# ---- video ----
SHORT_VIDEO_SEC = 30      # videos up to this length are sent to the model whole
CLIP_SEC        = 10      # length of each sampled clip for longer videos
MAX_CLIPS       = 6       # at most this many clips per video (speed vs. coverage knob)
CLIP_HEIGHT     = 360     # clips are downscaled to this height before decoding

# ---- SPEED PRESET: "fast" | "balanced" | "quality" ----
# Visual tokens ~ (nframes / 2) * (max_pixels / 784). Prefill time grows with it.
#   quality  : 24 frames @ 360x420 -> ~2300 visual tokens  (V2 behaviour)
#   balanced : 12 frames @ 280x280 -> ~600
#   fast     :  8 frames @ 224x224 -> ~260
PRESET = "fast"
SUMMARY_SENTENCES = 1     # shorter generated summary = fewer decode steps (1 is fastest, 2-3 = more detail)
VIDEO_MODE = "single"     # "single": ONE model call per video, frames spread over the whole video (fastest)
                          # "clips" : describe up to MAX_CLIPS clips, then summarize the notes (slower, more temporal detail)
MAX_FRAME_MULT = 3        # in "single" mode, longer videos get up to this many times the preset's frame count
PRESCALE_SHORT = False    # True: downscale + 4 fps via ffmpeg before decoding (helps if profile shows slow decode_media)

# Tiers from best to cheapest. On OOM the code steps down this list; STATE remembers what worked.
# min_pixels is set explicitly: newer qwen-omni-utils defaults it to ~100k pixels, which is larger
# than the max_pixels of the cheaper tiers and triggers "max_pixels must be >= min_pixels".
MIN_PX = 16 * 28 * 28
TIERS = [
    {"nframes": 24, "max_pixels": 360 * 420, "min_pixels": MIN_PX},   # quality
    {"nframes": 12, "max_pixels": 280 * 280, "min_pixels": MIN_PX},   # balanced
    {"nframes": 8,  "max_pixels": 224 * 224, "min_pixels": MIN_PX},   # fast
    {"nframes": 8,  "max_pixels": 168 * 168, "min_pixels": MIN_PX},   # emergency
]
_start = {"quality": 0, "balanced": 1, "fast": 2}[PRESET]
STATE = {"single_tier": _start, "clip_tier": max(_start, 1), "batch": 3}
FINAL_TOKENS = 30 + 30 * SUMMARY_SENTENCES   # output budget for TOPIC + SUMMARY

# ---- judge ----
JUDGE_THRESHOLD = 0.5     # a skill is matched when judge >= this. Tune it on a few labeled examples.
JUDGE_BATCH     = 8       # skills scored per forward pass (lower it if you hit OOM with many skills)
SHOW_EMBED      = False   # True adds an "embed" similarity column (needs sentence-transformers)

LABEL_EXAMPLES = "'Programming' (not 'Tower of Hanoi puzzle'), 'Chess' (not 'two players moving pieces'), 'Cooking', 'Dogs', 'Finance', 'Movies'"

SYSTEM = {"role": "system", "content": [{"type": "text",
    "text": "You are a concise assistant. Answer directly in plain text. Never ask the user questions."}]}

# One generation returns both the label and the summary
FINAL_PROMPT = ("Reply in exactly this format and nothing else:\n"
    "TOPIC: <1 or 2 words, the broad subject field, not a description. Examples: " + LABEL_EXAMPLES + ">\n"
    f"SUMMARY: <what this is about, at most {SUMMARY_SENTENCES} short sentences>")
CLIP_PROMPT = "Describe what happens in this clip in 2 short sentences."

In [23]:
def _cleanup():
    gc.collect(); torch.cuda.empty_cache()

PROFILE = {}   # seconds per stage (plus token counts), reset at the start of every check()
def _add(k, v): PROFILE[k] = PROFILE.get(k, 0) + v

def _generate(convs, max_new_tokens):
    """Run a batch of conversations through the thinker. Returns one string per conversation."""
    t0 = time.time()
    texts = processor.apply_chat_template(convs, add_generation_prompt=True, tokenize=False)
    audios, images, videos = process_mm_info(convs, use_audio_in_video=False)   # video decoding happens here
    t1 = time.time()
    inputs = processor(text=texts, audio=audios, images=images, videos=videos,
                       return_tensors="pt", padding=True, use_audio_in_video=False)
    inputs = inputs.to("cuda").to(torch.float16)
    torch.cuda.synchronize(); t2 = time.time()
    with torch.inference_mode():
        out = thinker.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    torch.cuda.synchronize(); t3 = time.time()
    n_in = inputs.input_ids.shape[1]
    _add("decode_media_s", t1 - t0); _add("preprocess_s", t2 - t1); _add("prefill+generate_s", t3 - t2)
    _add("input_tokens", n_in * len(convs)); _add("new_tokens", (out.shape[1] - n_in) * len(convs))
    return [t.strip() for t in processor.batch_decode(out[:, n_in:], skip_special_tokens=True)]

def generate_many(contents, max_new_tokens=200, tier_key="single_tier"):
    """
    contents : list of content-part lists (one per request). Processed in batches of STATE["batch"].
    On OOM: first halve the batch size, then drop to a cheaper video tier. Both are remembered.
    """
    results, i = [None] * len(contents), 0
    while i < len(contents):
        group = contents[i:i + STATE["batch"]]
        has_video = any(c["type"] == "video" for content in group for c in content)
        convs = []
        for content in group:
            content = [dict(c) for c in content]
            for c in content:
                if c["type"] == "video":
                    m = c.pop("frame_mult", 1)                       # longer video -> more frames (still even)
                    c.update(TIERS[STATE[tier_key]])
                    c["nframes"] = max(4, int(round(c["nframes"] * m / 2)) * 2)
            convs.append([SYSTEM, {"role": "user", "content": content}])
        try:
            results[i:i + len(group)] = _generate(convs, max_new_tokens)
            i += len(group)
        except torch.cuda.OutOfMemoryError:
            _cleanup()
            if STATE["batch"] > 1:
                STATE["batch"] = max(1, STATE["batch"] // 2)
                print(f"Out of memory -> batch size now {STATE['batch']}")
            elif has_video and STATE[tier_key] < len(TIERS) - 1:
                STATE[tier_key] += 1
                print(f"Out of memory -> video tier now {TIERS[STATE[tier_key]]}")
            else:
                results[i] = "(out of memory)"; i += 1
    _cleanup()
    return results

def run_text(prompt, max_new_tokens=200):
    return generate_many([[{"type": "text", "text": prompt}]], max_new_tokens)[0]

# Warm-up: the first generate call pays one-time CUDA/kernel setup. Do it now so timings below are honest.
_t = time.time(); run_text("Say OK.", 4); PROFILE.clear()
print(f"Warm-up done in {time.time() - _t:.1f}s")

Warm-up done in 1.5s


In [24]:
# ---- images, audio, PDF, docx, text ----
def _ext(src):
    """File extension for an existing path or a URL; '' for plain typed-in text."""
    if src.startswith("http") or os.path.exists(src):
        return os.path.splitext(src.split("?")[0])[1].lower()
    return ""

def _spread(items, k):
    """Pick up to k items evenly across the list (beginning, middle and end)."""
    if len(items) <= k: return items
    idx = np.unique(np.linspace(0, len(items) - 1, k).astype(int))
    return [items[i] for i in idx]

def _text_content(text):
    if len(text) > MAX_CHARS:             # long text: summarize pieces (batched), then the notes
        pieces = _spread([text[i:i + MAX_CHARS] for i in range(0, len(text), MAX_CHARS)], MAX_PIECES)
        notes = generate_many([[{"type": "text", "text": f"{p}\n\nSummarize this part in 3 sentences."}]
                               for p in pieces], 100)
        text = "\n".join(f"Part {i + 1}: {n}" for i, n in enumerate(notes))
    return [{"type": "text", "text": f"{text}\n\n{FINAL_PROMPT}"}]

def _pdf_content(path):
    with pymupdf.open(path) as doc:
        text = "\n".join(page.get_text() for page in doc)
        if len(text.strip()) > 200:
            return _text_content(text)
        content = []                       # scanned PDF: first 3 pages as images
        for i, page in enumerate(doc):
            if i >= 3: break
            png = f"/content/_page{i}.png"
            page.get_pixmap(dpi=100).save(png)
            content.append({"type": "image", "image": png})
    return content + [{"type": "text", "text": FINAL_PROMPT}]

def build_content(src):
    """Non-video input -> model-ready content parts."""
    ext = _ext(src)
    if not ext and not src.startswith("http"):
        return _text_content(src)          # plain text typed in directly
    if ext in IMG: return [{"type": "image", "image": src}, {"type": "text", "text": FINAL_PROMPT}]
    if ext in AUD: return [{"type": "audio", "audio": src}, {"type": "text", "text": FINAL_PROMPT}]
    if ext == ".pdf": return _pdf_content(src)
    if ext == ".docx": return _text_content("\n".join(p.text for p in docx.Document(src).paragraphs))
    if ext in TXT:
        with open(src, errors="ignore") as f: return _text_content(f.read())
    raise ValueError(f"Unsupported file type: {ext}")

In [25]:
# ---- video: sample a few clips, describe them in batches, then one final summary ----
def video_duration(path):
    r = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration",
                        "-of", "default=nw=1:nk=1", path], capture_output=True, text=True)
    return float(r.stdout.strip())

def clip_starts(duration):
    """Start times of up to MAX_CLIPS windows of CLIP_SEC, spread evenly across the video."""
    if duration <= MAX_CLIPS * CLIP_SEC:                       # short enough: contiguous coverage
        return [k * CLIP_SEC for k in range(math.ceil(duration / CLIP_SEC))]
    return [round((duration - CLIP_SEC) * k / (MAX_CLIPS - 1), 2) for k in range(MAX_CLIPS)]

def extract_clips(path, duration, out_dir="/content/clips"):
    """Cut only the sampled windows, downscaled and re-encoded fast (so cuts are exact, decode is cheap)."""
    os.makedirs(out_dir, exist_ok=True)
    for f in glob.glob(f"{out_dir}/*.mp4"): os.remove(f)
    starts = clip_starts(duration)
    def cut(k_s):
        k, s = k_s
        out = f"{out_dir}/clip_{k:02d}.mp4"
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-ss", str(s), "-t", str(CLIP_SEC), "-i", path,
                        "-an", "-vf", f"scale=-2:{CLIP_HEIGHT}", "-c:v", "libx264", "-preset", "ultrafast",
                        "-crf", "30", out], check=True)
        return out, s
    with ThreadPoolExecutor(max_workers=4) as ex:
        return list(ex.map(cut, enumerate(starts)))

def prescale(path, out="/content/prescaled.mp4"):
    """Optional: shrink a short video (low height, 4 fps) so frame decoding is cheap."""
    t = time.time()
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", path, "-an",
                    "-vf", f"scale=-2:{CLIP_HEIGHT},fps=4", "-c:v", "libx264", "-preset", "ultrafast",
                    "-crf", "30", out], check=True)
    _add("prescale_s", time.time() - t)
    return out

def _mmss(sec): return f"{int(sec // 60)}:{int(sec % 60):02d}"

def summarize_video(src):
    """Returns raw 'TOPIC: ...\nSUMMARY: ...' text."""
    duration = video_duration(src)
    if VIDEO_MODE == "single" or duration <= SHORT_VIDEO_SEC:
        video = prescale(src) if PRESCALE_SHORT else src
        mult = min(MAX_FRAME_MULT, max(1.0, duration / SHORT_VIDEO_SEC))
        content = [{"type": "video", "video": video, "frame_mult": mult}, {"type": "text", "text": FINAL_PROMPT}]
        return generate_many([content], FINAL_TOKENS, "single_tier")[0]
    _t = time.time()
    clips = extract_clips(src, duration)
    _add("extract_clips_s", time.time() - _t)
    notes = generate_many([[{"type": "video", "video": p}, {"type": "text", "text": CLIP_PROMPT}]
                           for p, _ in clips], 70, "clip_tier")
    joined = "\n".join(f"[{_mmss(s)}] {n}" for (_, s), n in zip(clips, notes))
    prompt = ("These are notes on clips sampled evenly across one video, with their timestamps:\n\n"
              f"{joined}\n\n{FINAL_PROMPT}")
    return generate_many([[{"type": "text", "text": prompt}]], FINAL_TOKENS, "clip_tier")[0]

def summarize(src):
    """Any input -> raw 'TOPIC: ...\nSUMMARY: ...' text."""
    if _ext(src) in VID: return summarize_video(src)
    return generate_many([build_content(src)], FINAL_TOKENS)[0]

In [26]:
tok = processor.tokenizer
def _ids(words): return sorted({tok.encode(w, add_special_tokens=False)[0] for w in words})
YES_IDS, NO_IDS = _ids(["YES", "Yes", "yes"]), _ids(["NO", "No", "no"])

JUDGE_SYSTEM = {"role": "system", "content": [{"type": "text",
    "text": "You are a strict classifier. Reply with exactly one word: YES or NO."}]}

def skill_text(skill): return f"{skill['name']}: {skill['description']}"

def clean_label(raw, max_words=3):
    label = raw.strip().split("\n")[0].strip(" \"'`*.:;-<>")
    if label.lower().startswith("label"):
        label = label.split(":", 1)[-1].strip(" \"'`*.")
    return " ".join(label.split()[:max_words])

def parse_output(raw):
    t = re.search(r"TOPIC:\s*(.+)", raw, re.I)
    s = re.search(r"SUMMARY:\s*(.+)", raw, re.I | re.S)
    return (clean_label(t.group(1)) if t else ""), (s.group(1).strip() if s else raw.strip())

def make_label(summary):
    """Fallback only: used when the combined generation did not return a TOPIC line."""
    q = (f"Content summary:\n{summary}\n\nWhat general subject or field is this content about? "
         f"Answer with 1 or 2 words only: the broad category, not a description. Examples: {LABEL_EXAMPLES}. "
         "Output only the category.")
    return clean_label(run_text(q, 8))

def judge_all(label, summary, skills):
    """P(YES) per skill from next-token logits: one batched forward pass per JUDGE_BATCH skills."""
    probs = []
    for i in range(0, len(skills), JUDGE_BATCH):
        prompts = []
        for s in skills[i:i + JUDGE_BATCH]:
            q = (f"Content topic: {label}\nContent summary: {summary}\n\nSkill: {skill_text(s)}\n\n"
                 "Is the content clearly about this skill's subject area, so that a person with this skill "
                 "would find it relevant? Answer YES or NO.")
            conv = [JUDGE_SYSTEM, {"role": "user", "content": [{"type": "text", "text": q}]}]
            prompts.append(processor.apply_chat_template(conv, add_generation_prompt=True, tokenize=False))
        inputs = processor(text=prompts, return_tensors="pt", padding=True, use_audio_in_video=False)
        inputs = inputs.to("cuda").to(torch.float16)
        with torch.inference_mode():
            try:    out = thinker(**inputs, use_cache=False, logits_to_keep=1)
            except TypeError: out = thinker(**inputs, use_cache=False)
            logits = out.logits[:, -1].float()
        yes, no = torch.logsumexp(logits[:, YES_IDS], 1), torch.logsumexp(logits[:, NO_IDS], 1)
        probs += torch.sigmoid(yes - no).tolist()
        del out, logits; _cleanup()
    return probs

_embedder = None
def embed_scores(summary, skills):
    global _embedder
    from sentence_transformers import SentenceTransformer, util
    if _embedder is None:
        _embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")
    e_s = _embedder.encode(summary, convert_to_tensor=True, normalize_embeddings=True)
    e_k = _embedder.encode([skill_text(s) for s in skills], convert_to_tensor=True, normalize_embeddings=True)
    return util.cos_sim(e_s, e_k)[0].tolist()

CACHE = {}   # (label, summary) per input, so re-checking with other skills/thresholds is nearly free
def _key(src):
    if os.path.exists(src):
        st = os.stat(src); return ("file", os.path.abspath(src), st.st_size, int(st.st_mtime))
    return ("str", hashlib.md5(src.encode()).hexdigest())

def check(src, skills, threshold=None, use_cache=True):
    """
    src    : text, or a path/URL to an image, video, audio, pdf, docx or text file
    skills : list of {"name": ..., "description": ...}
    returns: dict with related, matches, label, summary, table (per-skill scores), timing (seconds)
    """
    threshold = JUDGE_THRESHOLD if threshold is None else threshold
    timing, key = {}, _key(src)
    PROFILE.clear()
    t = time.time()
    if use_cache and key in CACHE:
        label, summary = CACHE[key]; timing["summarize"] = 0.0
    else:
        label, summary = parse_output(summarize(src))
        if not label: label = make_label(summary)
        CACHE[key] = (label, summary); timing["summarize"] = round(time.time() - t, 1)

    t = time.time()
    rows = [{"skill": s["name"], "judge": round(p, 3)} for s, p in zip(skills, judge_all(label, summary, skills))]
    table = pd.DataFrame(rows)
    if SHOW_EMBED: table["embed"] = [round(x, 3) for x in embed_scores(summary, skills)]
    table = table.sort_values("judge", ascending=False).reset_index(drop=True)
    table["matched"] = table.judge >= threshold
    timing["judge"] = round(time.time() - t, 1)
    matches = table[table.matched].skill.tolist()
    return {"related": bool(matches), "matches": matches, "label": label,
            "summary": summary, "table": table, "timing": timing,
            "profile": {k: round(v, 2) for k, v in PROFILE.items()}}

def show(r, verbose=False):
    print(f"Topic   : {r['label']}")
    print(f"Related : {r['related']}   Matched: {r['matches']}")
    print(f"Time (s): {r['timing']}")
    if r.get("profile"): print(f"Profile : {r['profile']}")
    if verbose: print(f"Summary : {r['summary']}")
    return r["table"]

In [27]:
# Test skills: what a user would define when creating a profile
TEST_SKILLS = [
    {"name": "Python Programming",  "description": "writing, debugging and explaining Python code, scripts and software"},
    {"name": "Cooking",             "description": "recipes, ingredients, food preparation and kitchen techniques"},
    {"name": "Pets & Animals",      "description": "dogs, cats, wildlife, animal behavior and pet care"},
    {"name": "Finance & Investing", "description": "stock markets, investing, banking, budgeting and economics"},
    {"name": "Movies & Cinema",     "description": "films, scenes, actors, directing and film analysis"},
    {"name": "Biology & Science",   "description": "biology, plants, chemistry, physics and natural science"},
]

In [35]:
# Try it: text
show(check("Today we are going to make pizza together. First, stretch the dough and add tomato sauce.", TEST_SKILLS))

Topic   : Cooking
Related : True   Matched: ['Cooking']
Time (s): {'summarize': 0.0, 'judge': 0.6}


,skill,judge,matched
0,Cooking,0.851,True
1,Biology & Science,0.246,False
2,Python Programming,0.072,False
3,Movies & Cinema,0.025,False
4,Pets & Animals,0.005,False
5,Finance & Investing,0.002,False


In [36]:
show(check("/content/30sec2.mp4", TEST_SKILLS))

Topic   : Feeding Animals
Related : True   Matched: ['Pets & Animals', 'Biology & Science']
Time (s): {'summarize': 0.0, 'judge': 0.5}


,skill,judge,matched
0,Pets & Animals,0.820,True
1,Biology & Science,0.660,True
2,Movies & Cinema,0.413,False
3,Python Programming,0.087,False
4,Cooking,0.068,False
5,Finance & Investing,0.003,False


In [37]:
show(check("/content/15sec2.mp4", TEST_SKILLS))

Topic   : Car Drifting
Related : False   Matched: []
Time (s): {'summarize': 0.0, 'judge': 0.6}


,skill,judge,matched
0,Movies & Cinema,0.479,False
1,Python Programming,0.076,False
2,Biology & Science,0.018,False
3,Pets & Animals,0.005,False
4,Finance & Investing,0.003,False
5,Cooking,0.002,False


In [31]:
show(check("/content/1min.mp4", TEST_SKILLS))

Topic   : Virtual Reality
Related : True   Matched: ['Python Programming']
Time (s): {'summarize': 4.3, 'judge': 0.5}
Profile : {'decode_media_s': 1.24, 'preprocess_s': 0.13, 'prefill+generate_s': 2.49, 'input_tokens': 1396, 'new_tokens': 19}


,skill,judge,matched
0,Python Programming,0.649,True
1,Movies & Cinema,0.277,False
2,Biology & Science,0.093,False
3,Cooking,0.017,False
4,Pets & Animals,0.016,False
5,Finance & Investing,0.015,False


In [32]:
show(check("/content/7_test1", TEST_SKILLS))

Topic   : Testing
Related : True   Matched: ['Python Programming']
Time (s): {'summarize': 1.4, 'judge': 0.6}
Profile : {'decode_media_s': 0.0, 'preprocess_s': 0.0, 'prefill+generate_s': 1.04, 'input_tokens': 120, 'new_tokens': 16}


,skill,judge,matched
0,Python Programming,0.637,True
1,Movies & Cinema,0.111,False
2,Biology & Science,0.028,False
3,Finance & Investing,0.012,False
4,Pets & Animals,0.005,False
5,Cooking,0.004,False


In [33]:
show(check("/content/movie_clip1.mp4", TEST_SKILLS))

Topic   : Education
Related : False   Matched: []
Time (s): {'summarize': 5.9, 'judge': 0.5}
Profile : {'decode_media_s': 2.46, 'preprocess_s': 0.28, 'prefill+generate_s': 2.49, 'input_tokens': 1762, 'new_tokens': 9}


,skill,judge,matched
0,Python Programming,0.159,False
1,Movies & Cinema,0.068,False
2,Biology & Science,0.027,False
3,Finance & Investing,0.012,False
4,Cooking,0.008,False
5,Pets & Animals,0.006,False


In [34]:
show(check("/content/programming.mp4", TEST_SKILLS))

Topic   : Tower of Hanoi
Related : True   Matched: ['Python Programming']
Time (s): {'summarize': 2.6, 'judge': 0.6}
Profile : {'decode_media_s': 0.17, 'preprocess_s': 0.05, 'prefill+generate_s': 1.97, 'input_tokens': 656, 'new_tokens': 23}


,skill,judge,matched
0,Python Programming,0.740,True
1,Movies & Cinema,0.112,False
2,Biology & Science,0.017,False
3,Cooking,0.005,False
4,Finance & Investing,0.004,False
5,Pets & Animals,0.004,False
